# Lab type: review
# Course: ML201 — Applied Machine Learning
# Lesson: Random Forests in Depth
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

np.random.seed(42)
n = 2000

income = np.random.normal(55000, 20000, n).clip(15000, 150000)
debt_ratio = np.random.beta(2, 5, n)
credit_score = np.random.normal(670, 80, n).clip(300, 850)
employment_years = np.random.exponential(5, n).clip(0, 40)
num_accounts = np.random.randint(1, 15, n)
region = np.random.randint(0, 5, n)
account_type = np.random.randint(0, 3, n)

log_odds = (
    -3.0
    + 1.5 * debt_ratio
    - 0.003 * (credit_score - 670) / 80
    - 0.00001 * income
    - 0.05 * employment_years
    + 0.02 * num_accounts
)
prob_default = 1 / (1 + np.exp(-log_odds))
target = (np.random.rand(n) < prob_default).astype(int)

feature_names = ['income', 'debt_ratio', 'credit_score', 'employment_years',
                 'num_accounts', 'region', 'account_type']
X = pd.DataFrame({
    'income': income,
    'debt_ratio': debt_ratio,
    'credit_score': credit_score,
    'employment_years': employment_years,
    'num_accounts': num_accounts,
    'region': region,
    'account_type': account_type
})
y = pd.Series(target, name='default')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Feature names:', feature_names)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')
print(f'Class balance (full dataset):\n{y.value_counts(normalize=True).round(3)}')

## Part 1: n_estimators and OOB Score

In [ ]:
n_values = list(range(10, 310, 20))
oob_scores = []

for n_trees in n_values:
    rf = RandomForestClassifier(
        n_estimators=n_trees,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    oob_scores.append(rf.oob_score_)

plt.figure(figsize=(9, 4))
plt.plot(n_values, oob_scores, marker='o', markersize=4)
plt.xlabel('n_estimators')
plt.ylabel('OOB Accuracy')
plt.title('OOB Score vs Number of Trees')
plt.tight_layout()
plt.show()

plateau_n = None
for i in range(1, len(oob_scores)):
    if abs(oob_scores[i] - oob_scores[i - 1]) < 0.001:
        plateau_n = n_values[i]
        break

print(f'OOB score plateaus (improvement < 0.001) at approximately n_estimators = {plateau_n}')

**Question 1:** The OOB score stops improving around n=120. Why does adding more trees beyond this point not hurt performance, even though it wastes compute? What does this tell you about the bias-variance tradeoff with `n_estimators`?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Why more trees don't hurt:** Each additional tree is trained independently on a fresh bootstrap sample. The ensemble prediction is the average of all tree predictions — adding more trees can only reduce variance (toward the law-of-large-numbers limit), never increase it or introduce bias. There is no overfitting with respect to `n_estimators`.

**What the plateau tells you about bias-variance:** Once you have enough trees that the averaged prediction has converged, further trees reduce the remaining variance toward zero — but there is almost none left to remove. The plateau means variance has already been reduced to its practical minimum for this model complexity. Unlike max_depth or max_features, `n_estimators` controls only variance, not bias; the bias is determined by the individual tree structure.

</details>

**Question 2:** The OOB score is described as approximately equivalent to leave-one-out CV. How is it computed internally? When would you still use proper cross-validation instead of relying on the OOB score?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q2</summary>

**How OOB is computed:** Each tree is trained on a bootstrap sample of ~63% of the training rows. The remaining ~37% (the "out-of-bag" rows) are used to evaluate that tree. For each training example, sklearn aggregates predictions only from trees whose bootstrap sample did NOT include that example — typically 37% of all trees. The final OOB score averages these per-example predictions across the entire training set.

**When to use proper CV instead:** Use CV when: (1) you need reliable probability calibration (OOB per-example predictions come from fewer trees, so confidence estimates are noisier); (2) you are comparing multiple model types (OOB is only available for bagging methods); (3) the number of trees is small (very few OOB predictors per example makes the estimate unreliable); or (4) your dataset is small and you need maximally stable variance estimates.

</details>

## Part 2: max_features and Tree Correlation

In [ ]:
max_features_options = [1, 'sqrt', 'log2', None]
labels = ['1', 'sqrt', 'log2', 'None (all)']

print(f"{'max_features':<15} {'OOB Accuracy':<15}")
print('-' * 30)
for mf, label in zip(max_features_options, labels):
    rf = RandomForestClassifier(
        n_estimators=200,
        max_features=mf,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    print(f"{label:<15} {rf.oob_score_:.4f}")

**Question 3:** `max_features=None` (all features at every split) produces the highest individual tree quality but typically lower ensemble performance. Why does using ALL features at every split undermine the ensemble?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Why all features undermines the ensemble:** When every tree sees all features at every split, every tree tends to split on the same dominant features in the same order — the trees become highly correlated with each other. The variance reduction from averaging n independent estimators is σ²/n; for correlated estimators with pairwise correlation ρ, variance reduces only to σ²(1/n + (n-1)ρ/n), which approaches σ²ρ as n grows. High inter-tree correlation means you are essentially averaging the same prediction many times, with minimal reduction in variance.

`max_features` is designed to decorrelate trees by forcing each split to consider only a random subset of features — this is the entire mechanism that makes random forests better than a single deep tree.

</details>

**Question 4:** When would you lower `max_features` below `'sqrt'`? What signal would tell you the current setting is wrong?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q4</summary>

**When to lower `max_features` below `'sqrt'`:** When you have many noisy or correlated features and the dominant signal is concentrated in a small number of them. Restricting to fewer features per split forces trees to occasionally "miss" the dominant features, ensuring they build alternative predictive structures — which reduces correlation more aggressively.

**Signal that the current setting is wrong:** A large gap between individual-tree OOB accuracy and the full ensemble OOB accuracy that closes when you lower `max_features`; permutation importance showing that 2–3 features completely dominate and the rest carry near-zero signal; or OOB scores that continue improving as you reduce `max_features` below `'sqrt'`.

</details>

## Part 3: MDI vs Permutation Importance

In [ ]:
# Add a customer_id column (sequential integers) — a textbook leaky-looking feature
X_train_aug = X_train.copy().reset_index(drop=True)
X_test_aug = X_test.copy().reset_index(drop=True)
X_train_aug['customer_id'] = np.arange(len(X_train_aug))
X_test_aug['customer_id'] = np.arange(len(X_train_aug), len(X_train_aug) + len(X_test_aug))

rf_full = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_full.fit(X_train_aug, y_train)

# MDI importance
mdi_importances = pd.Series(
    rf_full.feature_importances_,
    index=X_train_aug.columns
).sort_values(ascending=False)

# Permutation importance (on test set)
perm_result = permutation_importance(
    rf_full, X_test_aug, y_test,
    n_repeats=15,
    random_state=42,
    n_jobs=-1
)
perm_importances = pd.Series(
    perm_result.importances_mean,
    index=X_train_aug.columns
).sort_values(ascending=False)

# Side-by-side table
importance_df = pd.DataFrame({
    'MDI': mdi_importances,
    'Permutation': perm_importances
}).sort_values('MDI', ascending=False)

print('Feature Importances — MDI vs Permutation')
print(importance_df.round(4).to_string())

**Question 5:** `customer_id` ranks high in MDI importance but near-zero in permutation importance. Explain the mechanism: why does a sequential integer create high impurity reduction in tree splits, even though it has no real predictive relationship with the target?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q5</summary>

**The mechanism:** `customer_id` contains integers 0..n (near-unique values). At each split node, a decision tree evaluates all possible split thresholds on `customer_id`. With ~1000 unique values, the tree can find a threshold that separates a handful of positive-class examples from a large negative group — producing a high impurity reduction in that node, even though the split is completely arbitrary. MDI sums these spurious impurity reductions across all trees and all splits, giving `customer_id` an inflated importance score that reflects its cardinality, not its predictive value.

**Why permutation importance correctly scores it near zero:** At test time, shuffling `customer_id` changes the values (to a different permutation of a sequential range), but the model has no mechanism to use a specific integer value to predict the target — the model's AUC is unchanged. Permutation importance only captures real predictive signal.

</details>

**Question 6:** You are producing a feature importance report for a stakeholder who will use it to decide which data to collect next. Which importance measure do you use, and why?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q6</summary>

**Use permutation importance.** A stakeholder deciding which data to collect next needs to know which features genuinely help the model generalise to new cases — not which features happened to reduce training impurity. MDI is computed on training data and is biased by feature cardinality; it would direct the stakeholder to collect more `customer_id`-style identifiers, which carry no real signal. Permutation importance measures held-out predictive contribution and is unaffected by cardinality, making it the right measure for any decision about real-world data collection.

</details>

## Part 4: Partial Dependence Plots

In [ ]:
# Use the base model (without customer_id) for interpretable PDPs
rf_pdp = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_pdp.fit(X_train, y_train)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
PartialDependenceDisplay.from_estimator(
    rf_pdp,
    X_train,
    features=['income', 'credit_score'],
    kind='average',
    ax=ax
)
plt.suptitle('Partial Dependence Plots — income and credit_score', y=1.02)
plt.tight_layout()
plt.show()

**Question 7:** A PDP shows the expected model output as a function of one feature, with all other features "averaged out." For a feature that is highly correlated with another (e.g., `income` and `debt_ratio`), what limitation does the PDP have? What combinations of feature values might it be averaging over that would never appear in real data?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q7</summary>

**The limitation:** A PDP computes the average model prediction over all observed (or synthetic) values of the remaining features for each value of the focal feature. If `income` and `debt_ratio` are strongly correlated, the PDP averages over combinations like "very low income, very low debt_ratio" that rarely or never appear in the training data. The model may extrapolate arbitrarily in these regions, and the PDP average includes predictions on combinations the model was never trained on — producing a misleading marginal effect curve in unrealistic parts of the feature space.

</details>

**Question 8:** `kind='individual'` (ICE plots) shows one line per observation. When would ICE plots reveal something that the average PDP hides? Give a concrete example involving a feature that affects different subgroups differently.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q8</summary>

**When ICE reveals what the average PDP hides:** When different subgroups respond differently to a feature — i.e., there is a feature interaction. The average PDP line would be flat or show a mild trend, masking the fact that some lines go up while others go down.

**Concrete example:** Suppose `credit_score` increases default risk for customers with high `debt_ratio` (already over-leveraged customers taking on more credit with high scores may be underrepresented in data, so the model associates high score + high debt with risk), but decreases default risk for low `debt_ratio` customers. The average PDP line would be nearly flat. ICE plots would show two distinct bundles of lines crossing — a clear signal of an interaction between `credit_score` and `debt_ratio` that the PDP average obscures entirely.

</details>

## Summary

Check your understanding with these final questions. Each should be answerable in one sentence.

1. Why does the OOB error become a reliable performance estimate only once you have enough trees, and what does it mean for variance when you add more trees beyond the plateau?

2. In one sentence, explain why reducing `max_features` decreases tree correlation and why that generally improves ensemble generalisation.

3. In one sentence, state the key difference between MDI and permutation importance that makes MDI unreliable for high-cardinality or continuous features.

4. A PDP for `credit_score` shows a flat line between 600 and 750. Before concluding that credit score does not matter in this range, what alternative explanation should you check?

<details>
<summary>🔑 Reveal summary answers</summary>

1. **OOB reliability and variance:** With very few trees, each training example is predicted by only a handful of OOB trees, making the per-example estimate noisy and the aggregate OOB score unreliable; beyond the plateau, additional trees reduce the remaining variance asymptotically toward zero — the estimate converges and variance becomes negligible.

2. **max_features and tree correlation:** Restricting `max_features` means each tree only considers a random subset of features at each split, so different trees are forced to specialise on different subsets of predictors — reducing pairwise correlation ρ and thus the (n-1)ρ term that limits variance reduction when averaging an ensemble.

3. **MDI vs permutation:** MDI is computed on training splits and can be exploited by any feature with high cardinality or high numeric range to produce many fine-grained splits with large impurity reduction, while permutation importance measures actual held-out prediction impact and is insensitive to a feature's range or uniqueness.

4. **Flat PDP alternative explanation:** Check whether the training data contains examples with `credit_score` values in the 600–750 range — a flat PDP in a sparsely populated region may indicate the model is extrapolating over phantom data rather than a genuine absence of effect in that range.

</details>